# Lab 15 · Learn from feedback: a learning curve and per-model trust

**Prepared by Mustafa Assaf** · Self-study: Build your council further · 15–20 minute lab

## Show that the council improves with feedback

**So far:** Lab 07 stored a lesson and compared two held-out cases before and after. Lab 08 compared the council with single models.

**New:** the course project's second experiment, done properly, with a second learning lever:

1. **Split** a benchmark into *feedback* cases and *test* cases. The human's feedback comes only from the first; scores come only from the second.
2. **Per-model trust.** After each feedback case, record how often each architect agreed with the human. Weight each vote by how much better than chance that architect has been.
3. **A learning curve.** Score the test cases after every feedback round, not only before and after.

This lab needs no model calls: the architects' answers were recorded earlier, and simple rules stand in for them so the lab stays short. The cases are new; only the question is the same.

**At a glance**

![Lab 15 at a glance: Benchmark (10 cases), then Split once (6 feedback · 4 test), then Feedback case (one per round), then Update trust (per architect), then Weighted vote (test cases only), then Learning curve. Then back to Feedback case: next round.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/15.svg)

## The idea in context

A system that learns from feedback must show that it improves on new cases, not that it remembers old ones. Machine learning answered this long ago with a train/test split: learn from one set, score on another, and never mix them. A learning curve plots the score as feedback grows, which shows how fast the system learns and where it stalls.

Weighting voters by their record is an old idea too. The weighted majority algorithm (Littlestone and Warmuth, 1994) lowers an expert's weight each time it is wrong, so reliable experts come to dominate. Here, an architect that agrees with the human more often than chance gains influence, and one at chance level gets none.

| Key term | Meaning |
| --- | --- |
| **Train/test split** | Separate cases for learning and for scoring. |
| **Learning curve** | The score plotted after each round of feedback. |
| **Weighted vote** | A vote in which each voter counts according to its weight. |
| **Trust** | Here, how often an architect has agreed with the human on the feedback cases. |

## Pattern · Voting-Based Cooperation

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Independent perspectives reduce the chance that one model or reasoning path dominates the result. |
| **Problem** | One agent can be confidently wrong, and collecting several answers still leaves someone to reconcile them. |
| **Solution** | Ask several voters the same question independently, then select a result by majority, weighted scoring, or a judge. |
| **Use when** | Independent judgement is valuable; the task has a decision outcome; diverse personas or models are available. |
| **Trade-off** | Ensembles reduce individual error, but they can converge on a shared bias, and they cost N times as much. |
| **Flow** | Question → N voters → Aggregate → Decision |
| **Quality attributes** | Benefits: Reliability, Accuracy, Performance, Maintainability. Liabilities: Cost, Latency, Reliability, Accuracy. |
| **In this lab** | The aggregation is weighted scoring, and the weights are learned from each voter's record against the human. The guide's liability, a shared bias, shows in round 0: two voters share the same mistake and outvote the third. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. This lab needs no model key.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"

## Step 1 · A benchmark, split once

Ten small cases, six for feedback and four for testing. `expert` holds the human architect's decisions; here a rule stands in for them: the policy, plus the lesson that one operations engineer cannot run many services. The assert guards the split. In your project, shuffle with a fixed seed and keep the split for the whole experiment.

In [ ]:
scenarios = [  # (name, team_size, independent_deploys, ops_engineers)
    ("Library", 6, False, 1), ("Catalog", 9, True, 2), ("Chat", 18, True, 3), ("Ticketing", 16, True, 1),
    ("Ledger", 11, True, 1), ("Billing", 10, True, 3),
    ("Reports", 20, True, 1), ("Maps", 12, True, 2), ("Forms", 5, True, 2), ("Fleet", 17, True, 1),
]
expert = {n: "microservices" if t >= 8 and i and o >= 2 else "modular monolith" for n, t, i, o in scenarios}
feedback, test = [s[0] for s in scenarios[:6]], [s[0] for s in scenarios[6:]]
assert not set(feedback) & set(test), "A case must be a feedback case or a test case, never both."
print("feedback:", feedback, "\ntest:    ", test)

## Step 2 · Three architects, recorded

Each architect reads the facts differently. The Minimalist picks microservices only for very large teams, the Modernizer follows the policy alone, and the Safeguard also weighs the operations team, but slips once. Counting their test answers shows who is best, but the council must not use these numbers: it may learn only from the feedback cases.

In [ ]:
answers = {  # recorded answers; simple rules stand in for three models
    "Minimalist": {n: "microservices" if t >= 15 else "modular monolith" for n, t, i, o in scenarios},
    "Safeguard": {n: "microservices" if t >= 8 and i and o >= 2 else "modular monolith" for n, t, i, o in scenarios},
    "Modernizer": {n: "microservices" if t >= 8 and i else "modular monolith" for n, t, i, o in scenarios},
}
answers["Safeguard"]["Catalog"] = "modular monolith"  # even the best architect slips
for name, given in answers.items():
    print(f"{name:<11} right on {sum(given[n] == expert[n] for n in test)} of {len(test)} test cases")

## Step 3 · Trust and a weighted vote

`trust` is each architect's agreement with the human on the feedback cases seen so far. Adding 1 to the agreements and 2 to the cases keeps an architect with no record at 0.5, a coin toss. `weight` turns trust into a vote: 0 at 0.5, 1 at always right. `council` adds up the weights for each option, and a tie escalates to the human, as in Lab 06.

In [ ]:
def trust(seen):
    """How often each architect agreed with the human on the feedback cases seen so far, smoothed."""
    return {a: (sum(given[n] == expert[n] for n in seen) + 1) / (len(seen) + 2) for a, given in answers.items()}

def weight(p):
    return max(0.0, 2 * p - 1)  # how much better than a coin toss

def council(case, weights):
    """A weighted vote. A tie escalates to the human."""
    score = {"modular monolith": 0.0, "microservices": 0.0}
    for a, given in answers.items():
        score[given[case]] += weights[a]
    if round(score["modular monolith"], 9) == round(score["microservices"], 9):
        return "escalate"
    return max(score, key=score.get)

## Step 4 · The learning curve

Round 0 is the plain majority vote, before any feedback. After each round, one more feedback case is known, the trust is updated, and the four test cases are scored again. The final assert checks that feedback helped on cases it never saw.

In [ ]:
curve = []
for k in range(len(feedback) + 1):
    weights = {a: weight(p) for a, p in trust(feedback[:k]).items()} if k else dict.fromkeys(answers, 1.0)
    decisions = [council(n, weights) for n in test]
    curve.append(sum(d == expert[n] for d, n in zip(decisions, test)))
    shown = ", ".join(f"{a} {w:.2f}" for a, w in weights.items())
    print(f"round {k}: {curve[-1]}/{len(test)} correct  weights: {shown}  escalated: {decisions.count('escalate')}")
print("learning curve:", curve)
assert curve[-1] > curve[0], "Feedback did not improve the held-out score."

## What you should see

Step 2 shows the Safeguard right on 4 of 4 test cases, the Modernizer on 2, and the Minimalist on 1. In Step 4, the plain vote of round 0 gets 2 of 4: on Reports and Fleet, two architects choose microservices for a team with one operations engineer. Trust grows slowly: after round 4, Ticketing has taught the lesson, but the Safeguard and the Modernizer weigh the same, so two test cases escalate. From round 5 the Safeguard leads, and the council gets 4 of 4. The learning curve is `[2, 2, 2, 2, 2, 4, 4]`.

## Try it

1. Move Ticketing, the first feedback case about one operations engineer, to the front of the list. The curve rises at once, then falls back for two rounds before it recovers. Why is trust that rests on two or three cases fragile?
2. Use the trust itself as the weight (`return p` in `weight`). The curve stays flat, and the final assert stops the run. Why should an architect who is right half the time get no vote?
3. Delete the line with the Safeguard's slip. The curve now rises one round earlier.
4. Move Reports into the feedback set and score it anyway. Why would its score no longer count as evidence of learning?
5. **Use it in your council.** Plan your project's experiment: how many feedback and test scenarios, which learning lever (lessons in memory, principles, or per-model trust), and what the curve should show.

## Takeaway

Feedback must improve cases it never saw, so split the benchmark once and score only the test cases. Per-model trust is a learning lever that needs no prompt changes: it learns from outcomes which architect to believe. A learning curve shows how much feedback the council needs, and where it stalls.

## Read further

- [Littlestone and Warmuth, The weighted majority algorithm (1994)](https://doi.org/10.1006/inco.1994.1009)